In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

# Cell 8: Test Dataset — Data discovery

Once your Drive is mounted, please provide the full path to the data file you want to load (e.g., `/content/drive/MyDrive/path/to/your/file.csv`).

In [2]:

import glob
import os
import pandas as pd

KDD_DATA_PATH = None

# Search common Kaggle input locations
kdd_candidates = glob.glob('/kaggle/input/**/kdd*.csv', recursive=True)
if kdd_candidates:
    kdd_candidates.sort(key=os.path.getsize, reverse=True)
    KDD_DATA_PATH = kdd_candidates[0]
    print('Auto-detected KDD:', KDD_DATA_PATH)

# Fallback to repo data path
if KDD_DATA_PATH is None:
    for p in [
        '/content/drive/MyDrive/IDS dataset/kddcup99_csv.csv',
        '../data/KDD/kddcup99_csv.csv',
        '/kaggle/working/IDS_Interpretability/data/KDD/kddcup99_csv.csv',
    ]:
        if os.path.exists(p):
            KDD_DATA_PATH = p
            break

if KDD_DATA_PATH is None:
    print('⚠ KDD Cup 99 CSV not found — skipping cross-dataset evaluation.')
    print('  Upload the dataset to /kaggle/input/ or place it at data/KDD/kddcup99_csv.csv')
else:
    print('Using KDD dataset:', KDD_DATA_PATH)
    print('Size (MB):', os.path.getsize(KDD_DATA_PATH) / 1024**2)
    df_kdd_peek = pd.read_csv(KDD_DATA_PATH, nrows=5)
    label_cands = [c for c in df_kdd_peek.columns if 'label' in c.lower()]
    print('Columns:', len(df_kdd_peek.columns))
    print('Label col:', label_cands[0] if label_cands else 'NOT FOUND')
    print('Sample labels:', df_kdd_peek[label_cands[0]].tolist() if label_cands else [])

⚠ KDD Cup 99 CSV not found — skipping cross-dataset evaluation.
  Upload the dataset to /kaggle/input/ or place it at data/KDD/kddcup99_csv.csv


In [3]:
import torch

def setup_device():
    if torch.cuda.is_available():
        device_count = torch.cuda.device_count()
        for idx in range(device_count):
            gpu_name = torch.cuda.get_device_name(idx)
            memory_gb = torch.cuda.get_device_properties(idx).total_memory / 1024 ** 3
            print(f"GPU {idx}: {gpu_name} ({memory_gb:.1f} GB)")
        device = torch.device("cuda:0")
        return device, device_count > 1
    print("CUDA not available, using CPU")
    return torch.device("cpu"), False

def detect_label_column(df: pd.DataFrame) -> str:
    for col in df.columns:
        if "label" in col.lower():
            return col
    raise ValueError("Label column not found in dataset")

def _detect_normal_label(df: pd.DataFrame, label_col: str) -> str:
    """Return the value that represents 'normal/benign' traffic in the label column."""
    sample = df[label_col].dropna().unique()
    for candidate in ("BENIGN", "benign", "Benign"):
        if candidate in sample:
            return candidate
    for candidate in ("normal", "Normal", "NORMAL"):
        if candidate in sample:
            return candidate
    raise ValueError(
        f"Cannot auto-detect the normal/benign class in column '{label_col}'. "
        f"Unique values (first 20): {list(sample[:20])}"
    )


def prepare_features(df: pd.DataFrame, label_col: str):
    """Extract features as a float32 numpy array to minimise memory.

    Supports both CICIDS-2017 (BENIGN label, string IP cols) and
    KDD Cup 99 (normal label, categorical protocol/service/flag cols).
    """
    normal_value = _detect_normal_label(df, label_col)
    binary_label = (df[label_col] != normal_value).astype(np.int8).values

    # Drop columns that are identifiers / non-numeric metadata
    blacklist = {label_col, "Flow ID", "Source IP", "Destination IP", "Timestamp"}
    feature_cols = [col for col in df.columns if col not in blacklist]
    X = df[feature_cols].copy()

    # Label-encode remaining object/category columns (e.g. KDD protocol_type, service, flag)
    from sklearn.preprocessing import LabelEncoder
    obj_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    for col in obj_cols:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))

    X_np = X.values.astype(np.float32)
    np.nan_to_num(X_np, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    return X_np, binary_label, feature_cols


In [4]:
import torch
import torch.nn as nn


class CNNTokenizer(nn.Module):
    def __init__(self, input_dim: int, conv_channels: int, d_model: int):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, conv_channels, kernel_size=5, padding=2),
            nn.BatchNorm1d(conv_channels),
            nn.GELU(),
            nn.Conv1d(conv_channels, conv_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(conv_channels),
            nn.GELU(),
            nn.Dropout(0.1),
        )
        self.projection = nn.Linear(conv_channels, d_model)
        self.norm = nn.LayerNorm(d_model)
        self.input_dim = input_dim

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.unsqueeze(1)
        tokens = self.conv(x).transpose(1, 2)
        tokens = self.projection(tokens)
        return self.norm(tokens)


class CNNTransformerIDS(nn.Module):
    def __init__(
        self,
        input_dim: int,
        d_model: int,
        conv_channels: int,
        num_layers: int,
        num_heads: int,
        d_ff: int,
        dropout: float,
    ):
        super().__init__()
        self.tokenizer = CNNTokenizer(input_dim, conv_channels, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        self.positional = nn.Parameter(torch.randn(1, input_dim + 1, d_model))
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 2),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        tokens = self.tokenizer(x)
        batch_size, seq_len, _ = tokens.size()
        cls = self.cls_token.expand(batch_size, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        tokens = tokens + self.positional[:, : seq_len + 1]
        encoded = self.encoder(self.dropout(tokens))
        logits = self.classifier(encoded[:, 0])
        return logits


In [5]:
import torch.nn.functional as F
import numpy as np
def calculate_comprehensive_metrics(y_true, y_pred, y_prob):
    if len(y_true) == 0:
        return {key: 0.0 for key in ["accuracy", "auc_roc", "auc_pr", "f1_score", "precision", "recall"]}
    accuracy = float(np.mean(y_true == y_pred))
    unique_classes = np.unique(y_true)
    if len(unique_classes) < 2:
        return {
            "accuracy": accuracy,
            "auc_roc": 0.5,
            "auc_pr": float(np.mean(y_true)),
            "f1_score": 0.0,
            "precision": 0.0,
            "recall": 0.0,
        }

    from sklearn.metrics import (
        roc_auc_score,
        precision_recall_curve,
        auc,
        f1_score,
        precision_score,
        recall_score,
    )

    auc_roc = roc_auc_score(y_true, y_prob)
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    auc_pr = auc(recall, precision)
    f1 = f1_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    return {
        "accuracy": accuracy,
        "auc_roc": auc_roc,
        "auc_pr": auc_pr,
        "f1_score": f1,
        "precision": prec,
        "recall": rec,
    }


def binary_predictions_from_proba(y_prob: np.ndarray, threshold: float = 0.5) -> np.ndarray:
    probs = np.asarray(y_prob, dtype=np.float64)
    return (probs >= float(threshold)).astype(np.int64)

def _eval_epoch_with_threshold(model, loader, criterion, device, threshold: float):
    model.eval()
    losses = []
    all_probs, all_targets = [], []
    with torch.no_grad():
        for data, target in loader:
            data = data.to(device, non_blocking=True)
            target = target.to(device, non_blocking=True)
            logits = model(data)
            loss = criterion(logits, target)
            losses.append(loss.item())
            probs = F.softmax(logits, dim=1)[:, 1]
            all_probs.append(probs.detach().cpu().numpy())
            all_targets.append(target.detach().cpu().numpy())

    if not all_probs:
        return (
            0.0,
            {key: 0.0 for key in ["accuracy", "auc_roc", "auc_pr", "f1_score", "precision", "recall"]},
            np.array([]),
            np.array([]),
        )

    y_prob = np.concatenate(all_probs)
    y_true = np.concatenate(all_targets)
    y_pred = binary_predictions_from_proba(y_prob, threshold=threshold)
    metrics = calculate_comprehensive_metrics(y_true, y_pred, y_prob)
    return float(np.mean(losses)) if losses else 0.0, metrics, y_prob, y_true

In [6]:
import numpy as np

from sklearn.preprocessing import StandardScaler
import gc
from torch.utils.data import DataLoader, TensorDataset

In [7]:
def save_feature_comparison(train_feature_cols, target_feature_cols, output_dir):
    """Print and save feature comparison between train and test datasets."""
    os.makedirs(output_dir, exist_ok=True)

    train_set = set(train_feature_cols)
    test_set = set(target_feature_cols)

    common_features = sorted(train_set & test_set)
    train_only = sorted(train_set - test_set)
    test_only = sorted(test_set - train_set)

    print("\n" + "=" * 80)
    print("FEATURE COMPARISON")
    print("=" * 80)

    print(f"\nTrain features ({len(train_feature_cols)}):")
    for i, feat in enumerate(train_feature_cols, 1):
        print(f"  {i:3d}. {feat}")

    print(f"\nTest features ({len(target_feature_cols)}):")
    for i, feat in enumerate(target_feature_cols, 1):
        print(f"  {i:3d}. {feat}")

    print(f"\nCommon features ({len(common_features)}):")
    for i, feat in enumerate(common_features, 1):
        print(f"  {i:3d}. {feat}")

    print(f"\nTrain-only features ({len(train_only)}):")
    for i, feat in enumerate(train_only, 1):
        print(f"  {i:3d}. {feat}")

    print(f"\nTest-only features ({len(test_only)}):")
    for i, feat in enumerate(test_only, 1):
        print(f"  {i:3d}. {feat}")

    # Save side-by-side index-based comparison
    max_len = max(len(train_feature_cols), len(target_feature_cols))
    comparison_rows = []
    for i in range(max_len):
        train_feat = train_feature_cols[i] if i < len(train_feature_cols) else ""
        test_feat = target_feature_cols[i] if i < len(target_feature_cols) else ""
        comparison_rows.append({
            "index": i,
            "train_feature": train_feat,
            "test_feature": test_feat,
            "exact_match": train_feat == test_feat if train_feat and test_feat else False
        })

    comparison_df = pd.DataFrame(comparison_rows)
    comparison_path = os.path.join(output_dir, "feature_index_comparison.csv")
    comparison_df.to_csv(comparison_path, index=False)

    # Save grouped summary
    summary_max_len = max(len(common_features), len(train_only), len(test_only))
    summary_rows = []
    for i in range(summary_max_len):
        summary_rows.append({
            "common_feature": common_features[i] if i < len(common_features) else "",
            "train_only_feature": train_only[i] if i < len(train_only) else "",
            "test_only_feature": test_only[i] if i < len(test_only) else "",
        })

    summary_df = pd.DataFrame(summary_rows)
    summary_path = os.path.join(output_dir, "feature_set_summary.csv")
    summary_df.to_csv(summary_path, index=False)

    print(f"\nSaved feature index comparison -> {comparison_path}")
    print(f"Saved feature set summary      -> {summary_path}")
    print("=" * 80 + "\n")

    return common_features, train_only, test_only

In [8]:
from dataclasses import dataclass


@dataclass
class CNNTransformerConfig:
    input_path: str = "data/cicids2017/cicids2017.csv"
    output_dir: str = "artifacts"
    val_size: float = 0.1
    test_size: float = 0.2
    random_state: int = 42
    epochs: int = 25
    batch_size: int = 1024
    val_batch_size: int = 2048
    lr: float = 1.5e-3
    weight_decay: float = 1e-4
    label_smoothing: float = 0.05
    conv_channels: int = 96
    num_layers: int = 3
    num_heads: int = 8
    d_model: int = 192
    d_ff: int = 768
    dropout: float = 0.2
    undersampling_ratio: float = 0.15
    ig_steps: int = 32
    ig_samples: int = 512
    num_workers: int = 2
    max_train_samples: int = 500_000  # cap training set after balancing


In [9]:
def cross_dataset_evaluate(
    checkpoint_path: str,
    test_data_path: str,
    output_dir: str,
    batch_size: int = 2048,
    num_workers: int = 2,
    max_test_samples: int = 0,
    random_state: int = 42,
):
    """Evaluate a trained CNN-Transformer checkpoint on a different dataset.

    Cross-dataset evaluation logic:
    - map target/test features to training feature order
    - fill missing features using training medians from checkpoint
    - scale using training mean/scale from checkpoint
    """

    import re

    def normalize_feature_name(name: str) -> str:
        if pd.isna(name):
            return ""

        s = str(name).strip().lower()

        replacements = {
            "destination": "dst",
            "source": "src",
            "forward": "fwd",
            "backward": "bwd",
            "packet": "pkt",
            "packets": "pkts",
            "bytes": "byts",
            "length": "len",
            "count": "cnt",
            "average": "avg",
            "segment": "seg",
            "bulk": "blk",
            "total length of": "totlen",
            "total": "tot",
        }

        for old, new in replacements.items():
            s = s.replace(old, new)

        s = re.sub(r"[^a-z0-9/]+", " ", s)
        s = re.sub(r"\s+", " ", s).strip()
        return s

    manual_aliases = {
        "Destination Port": "Dst Port",
        "Flow Bytes/s": "Flow Byts/s",
        "Flow Packets/s": "Flow Pkts/s",

        "Total Fwd Packets": "Tot Fwd Pkts",
        "Total Backward Packets": "Tot Bwd Pkts",
        "Total Length of Fwd Packets": "TotLen Fwd Pkts",
        "Total Length of Bwd Packets": "TotLen Bwd Pkts",

        "Fwd Packet Length Max": "Fwd Pkt Len Max",
        "Fwd Packet Length Min": "Fwd Pkt Len Min",
        "Fwd Packet Length Mean": "Fwd Pkt Len Mean",
        "Fwd Packet Length Std": "Fwd Pkt Len Std",

        "Bwd Packet Length Max": "Bwd Pkt Len Max",
        "Bwd Packet Length Min": "Bwd Pkt Len Min",
        "Bwd Packet Length Mean": "Bwd Pkt Len Mean",
        "Bwd Packet Length Std": "Bwd Pkt Len Std",

        "Fwd Packets/s": "Fwd Pkts/s",
        "Bwd Packets/s": "Bwd Pkts/s",

        "Bwd IAT Total": "Bwd IAT Tot",
        "Fwd IAT Total": "Fwd IAT Tot",

        "Fwd Header Length": "Fwd Header Len",
        "Bwd Header Length": "Bwd Header Len",

        "ACK Flag Count": "ACK Flag Cnt",
        "ECE Flag Count": "ECE Flag Cnt",
        "FIN Flag Count": "FIN Flag Cnt",
        "PSH Flag Count": "PSH Flag Cnt",
        "RST Flag Count": "RST Flag Cnt",
        "SYN Flag Count": "SYN Flag Cnt",
        "URG Flag Count": "URG Flag Cnt",

        "Subflow Fwd Bytes": "Subflow Fwd Byts",
        "Subflow Bwd Bytes": "Subflow Bwd Byts",
        "Subflow Fwd Packets": "Subflow Fwd Pkts",
        "Subflow Bwd Packets": "Subflow Bwd Pkts",

        "Avg Fwd Segment Size": "Fwd Seg Size Avg",
        "Avg Bwd Segment Size": "Bwd Seg Size Avg",
        "Average Packet Size": "Pkt Size Avg",

        "Packet Length Mean": "Pkt Len Mean",
        "Packet Length Std": "Pkt Len Std",
        "Packet Length Variance": "Pkt Len Var",
        "Max Packet Length": "Pkt Len Max",
        "Min Packet Length": "Pkt Len Min",

        "Init_Win_bytes_forward": "Init Fwd Win Byts",
        "Init_Win_bytes_backward": "Init Bwd Win Byts",

        "act_data_pkt_fwd": "Fwd Act Data Pkts",
        "min_seg_size_forward": "Fwd Seg Size Min",

        "Fwd Avg Bulk Rate": "Fwd Blk Rate Avg",
        "Bwd Avg Bulk Rate": "Bwd Blk Rate Avg",
        "Fwd Avg Bytes/Bulk": "Fwd Byts/b Avg",
        "Bwd Avg Bytes/Bulk": "Bwd Byts/b Avg",
        "Fwd Avg Packets/Bulk": "Fwd Pkts/b Avg",
        "Bwd Avg Packets/Bulk": "Bwd Pkts/b Avg",
    }

    def convert_preprocessor_vector_to_dict(values, feature_cols, name):
        """Convert saved checkpoint preprocessor values into feature->value dict."""
        if values is None:
            raise ValueError(f"Checkpoint missing preprocessor.{name}")

        if isinstance(values, dict):
            return {str(k): float(v) for k, v in values.items()}

        if isinstance(values, (list, tuple, np.ndarray)):
            if len(values) != len(feature_cols):
                raise ValueError(
                    f"preprocessor.{name} length {len(values)} does not match "
                    f"feature_columns length {len(feature_cols)}"
                )
            return {feature_cols[i]: float(values[i]) for i in range(len(feature_cols))}

        raise TypeError(f"Unsupported type for preprocessor.{name}: {type(values)}")

    def build_feature_mapping(train_feature_cols, target_feature_cols):
        """Map training feature -> target/test feature."""
        target_set = set(target_feature_cols)
        target_norm_map = {normalize_feature_name(col): col for col in target_feature_cols}

        feature_map = {}
        mapping_rows = []

        for train_feat in train_feature_cols:
            mapped_feat = None
            map_type = "missing"

            if train_feat in target_set:
                mapped_feat = train_feat
                map_type = "exact"
            elif train_feat in manual_aliases:
                alias = manual_aliases[train_feat]
                if alias in target_set:
                    mapped_feat = alias
                    map_type = "manual_alias"

            if mapped_feat is None:
                norm_name = normalize_feature_name(train_feat)
                if norm_name in target_norm_map:
                    mapped_feat = target_norm_map[norm_name]
                    map_type = "normalized"

            feature_map[train_feat] = mapped_feat
            mapping_rows.append({
                "train_feature": train_feat,
                "mapped_test_feature": mapped_feat if mapped_feat is not None else "",
                "mapping_type": map_type,
                "is_mapped": mapped_feat is not None,
            })

        return feature_map, pd.DataFrame(mapping_rows)

    def align_and_fill_target_features(
        X_raw,
        target_feature_cols,
        train_feature_cols,
        feature_map,
        train_feature_medians,
    ):
        """
        Build target matrix in exact training feature order.
        Missing/unmapped features are filled with training medians (raw space).
        """
        X_df = pd.DataFrame(X_raw, columns=target_feature_cols)
        n_samples = len(X_df)
        n_features = len(train_feature_cols)

        X_aligned = np.zeros((n_samples, n_features), dtype=np.float32)

        for j, train_feat in enumerate(train_feature_cols):
            mapped_test_feat = feature_map.get(train_feat)

            if mapped_test_feat is not None and mapped_test_feat in X_df.columns:
                X_aligned[:, j] = X_df[mapped_test_feat].to_numpy(dtype=np.float32)
            else:
                fill_val = float(train_feature_medians.get(train_feat, 0.0))
                X_aligned[:, j] = fill_val

        return X_aligned

    def apply_training_scaler(X_raw_aligned, train_feature_cols, train_feature_means, train_feature_scales):
        """
        Apply the training StandardScaler parameters:
            x' = (x - mean) / scale
        """
        mean_vec = np.array([train_feature_means[col] for col in train_feature_cols], dtype=np.float32)
        scale_vec = np.array([train_feature_scales[col] for col in train_feature_cols], dtype=np.float32)

        # Avoid divide-by-zero if any saved scale is zero
        scale_vec = np.where(scale_vec == 0.0, 1.0, scale_vec)

        X_scaled = (X_raw_aligned - mean_vec) / scale_vec
        return X_scaled.astype(np.float32)

    device, _ = setup_device()
    os.makedirs(output_dir, exist_ok=True)

    # ── Load checkpoint ─────────────────────────────────────────────
    ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    model_state = ckpt["model_state_dict"]
    cfg = ckpt.get("config", {})
    train_feature_cols = ckpt.get("feature_columns", [])
    threshold = ckpt.get("best_threshold", 0.5)
    preprocessor = ckpt.get("preprocessor", {})

    if not train_feature_cols:
        raise ValueError("Checkpoint missing 'feature_columns'.")

    train_feature_medians = convert_preprocessor_vector_to_dict(
        preprocessor.get("medians"), train_feature_cols, "medians"
    )
    train_feature_means = convert_preprocessor_vector_to_dict(
        preprocessor.get("mean"), train_feature_cols, "mean"
    )
    train_feature_scales = convert_preprocessor_vector_to_dict(
        preprocessor.get("scale"), train_feature_cols, "scale"
    )

    # Infer input_dim from positional embedding
    pos_shape = model_state["positional"].shape
    model_input_dim = pos_shape[1] - 1

    if len(train_feature_cols) != model_input_dim:
        raise ValueError(
            f"Checkpoint mismatch: len(feature_columns)={len(train_feature_cols)} "
            f"but model_input_dim={model_input_dim}"
        )

    print(f"Model trained on {len(train_feature_cols)} features (input_dim={model_input_dim})")
    print(f"Using threshold: {threshold:.4f}")

    # ── Load target dataset ────────────────────────────────────────
    print(f"Loading target dataset: {test_data_path}")
    df = pd.read_csv(test_data_path, low_memory=False)
    label_col = detect_label_column(df)
    X_raw, y, target_feature_cols = prepare_features(df, label_col)
    del df
    gc.collect()

    print(f"Target dataset: {len(y)} samples, {X_raw.shape[1]} features")
    print(f"Attack ratio: {y.mean():.4f} ({y.sum()} attacks / {len(y)} total)")

    # Optional subsampling
    if max_test_samples > 0 and len(y) > max_test_samples:
        rng = np.random.RandomState(random_state)
        idx = rng.choice(len(y), size=max_test_samples, replace=False)
        X_raw = X_raw[idx]
        y = y[idx]
        print(f"Sub-sampled to {max_test_samples} rows")

    # ── Feature mapping ────────────────────────────────────────────
    feature_map, mapping_df = build_feature_mapping(train_feature_cols, target_feature_cols)

    mapping_path = os.path.join(output_dir, "resolved_feature_mapping.csv")
    mapping_df.to_csv(mapping_path, index=False)

    mapped_count = int(mapping_df["is_mapped"].sum())
    exact_count = int((mapping_df["mapping_type"] == "exact").sum())
    alias_count = int((mapping_df["mapping_type"] == "manual_alias").sum())
    norm_count = int((mapping_df["mapping_type"] == "normalized").sum())
    missing_count = int((mapping_df["mapping_type"] == "missing").sum())

    print("\n" + "=" * 80)
    print("FEATURE MAPPING SUMMARY")
    print("=" * 80)
    print(f"Total training features : {len(train_feature_cols)}")
    print(f"Mapped features         : {mapped_count}")
    print(f"  Exact matches         : {exact_count}")
    print(f"  Manual alias matches  : {alias_count}")
    print(f"  Normalized matches    : {norm_count}")
    print(f"Missing / median-filled : {missing_count}")
    print(f"Saved mapping report -> {mapping_path}")
    print("=" * 80 + "\n")

    # ── Align in training feature order, fill missing with train medians ──
    X_aligned_raw = align_and_fill_target_features(
        X_raw=X_raw,
        target_feature_cols=target_feature_cols,
        train_feature_cols=train_feature_cols,
        feature_map=feature_map,
        train_feature_medians=train_feature_medians,
    )
    del X_raw
    gc.collect()

    print(f"Aligned raw target matrix shape: {X_aligned_raw.shape}")

    # ── Apply training scaler from checkpoint ──────────────────────
    X_scaled = apply_training_scaler(
        X_raw_aligned=X_aligned_raw,
        train_feature_cols=train_feature_cols,
        train_feature_means=train_feature_means,
        train_feature_scales=train_feature_scales,
    )
    del X_aligned_raw
    gc.collect()

    if X_scaled.shape[1] != model_input_dim:
        raise ValueError(
            f"Aligned feature dimension {X_scaled.shape[1]} does not match model_input_dim {model_input_dim}"
        )

    # ── Build model and dataloader ─────────────────────────────────
    model = CNNTransformerIDS(
        input_dim=model_input_dim,
        d_model=cfg.get("d_model", 192),
        conv_channels=cfg.get("conv_channels", 96),
        num_layers=cfg.get("num_layers", 3),
        num_heads=cfg.get("num_heads", 8),
        d_ff=cfg.get("d_ff", 768),
        dropout=cfg.get("dropout", 0.2),
    ).to(device)

    model.load_state_dict(model_state, strict=True)
    model.eval()

    test_dataset = TensorDataset(
        torch.FloatTensor(X_scaled),
        torch.LongTensor(y),
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=num_workers > 0,
    )

    # ── Run evaluation ─────────────────────────────────────────────
    criterion = nn.CrossEntropyLoss()
    test_loss, test_metrics, _, _ = _eval_epoch_with_threshold(
        model, test_loader, criterion, device, threshold=threshold,
    )

    train_preview = ", ".join(train_feature_cols[:10])
    test_preview = ", ".join(target_feature_cols[:10])

    print(
        f"\n{'='*60}\n"
        f"CROSS-DATASET TEST RESULTS\n"
        f"  Train data features: {len(train_feature_cols)} ({train_preview}...)\n"
        f"  Test data features:  {len(target_feature_cols)} ({test_preview}...)\n"
        f"  Mapped features:     {mapped_count} / {len(train_feature_cols)}\n"
        f"  Exact matches:       {exact_count}\n"
        f"  Alias matches:       {alias_count}\n"
        f"  Normalized matches:  {norm_count}\n"
        f"  Missing / filled:    {missing_count}\n"
        f"{'='*60}\n"
        f"  Loss:      {test_loss:.4f}\n"
        f"  Threshold: {threshold:.4f}\n"
        f"  ROC-AUC:   {test_metrics['auc_roc']:.4f}\n"
        f"  PR-AUC:    {test_metrics['auc_pr']:.4f}\n"
        f"  F1-Score:  {test_metrics['f1_score']:.4f}\n"
        f"  Precision: {test_metrics['precision']:.4f}\n"
        f"  Recall:    {test_metrics['recall']:.4f}\n"
        f"  Accuracy:  {test_metrics['accuracy']:.4f}\n"
        f"{'='*60}"
    )

    results = {
        "checkpoint": checkpoint_path,
        "test_data": test_data_path,
        "train_features": len(train_feature_cols),
        "test_features": len(target_feature_cols),
        "model_input_dim": model_input_dim,
        "threshold": threshold,
        "mapped_features": mapped_count,
        "exact_matches": exact_count,
        "alias_matches": alias_count,
        "normalized_matches": norm_count,
        "missing_filled_with_train_median": missing_count,
        **test_metrics,
    }

    results_df = pd.DataFrame([results])
    results_path = os.path.join(output_dir, "cross_dataset_results.csv")
    results_df.to_csv(results_path, index=False)
    print(f"Saved results -> {results_path}")

    return test_metrics

In [10]:
# Cell 3: Configure CNN-Transformer

DATA_PATH = '/kaggle/input/datasets/towhidultonmoy/cicids2018/Friday-16-02-2018_TrafficForML_CICFlowMeter.csv'
KDD_DATA_PATH = DATA_PATH
USE_SAMPLE = ('sample' in DATA_PATH.lower())

cnn_cfg = CNNTransformerConfig(
    input_path=DATA_PATH,
    output_dir='/kaggle/working/',
    epochs=25 if not USE_SAMPLE else 5,
    batch_size=1024 if not USE_SAMPLE else 64,
    val_batch_size=2048 if not USE_SAMPLE else 128,
    lr=1.5e-3,
    num_workers=2,
    d_model=192 if not USE_SAMPLE else 64,
    conv_channels=96 if not USE_SAMPLE else 32,
    num_layers=3 if not USE_SAMPLE else 1,
    num_heads=8 if not USE_SAMPLE else 4,
    d_ff=768 if not USE_SAMPLE else 256,
    ig_steps=32 if not USE_SAMPLE else 8,
    ig_samples=512 if not USE_SAMPLE else 128,
    max_train_samples=500_000 if not USE_SAMPLE else 0,
)

print('Mode:', 'SAMPLE' if USE_SAMPLE else 'FULL')
print('Output dir:', cnn_cfg.output_dir)
print('Config:', cnn_cfg.epochs, 'epochs | d_model', cnn_cfg.d_model, '| batch', cnn_cfg.batch_size)


Mode: FULL
Output dir: /kaggle/working/
Config: 25 epochs | d_model 192 | batch 1024


In [11]:
cnn_path = '/kaggle/input/models/samaraweeramethun/training-2017/pytorch/default/1/cnn_transformer_ids.pth'
# Cell 9: Cross-dataset evaluation — CICIDS-trained model on KDD Cup 99
if KDD_DATA_PATH is not None and cnn_path is not None:
    # from cnn_transformer_only.training.cnn_trainer import cross_dataset_evaluate

    cross_out = os.path.join(cnn_cfg.output_dir, 'cross_dataset_kdd')
    os.makedirs(cross_out, exist_ok=True)

    cross_metrics = cross_dataset_evaluate(
        checkpoint_path=cnn_path,
        test_data_path=KDD_DATA_PATH,
        output_dir=cross_out,
        batch_size=2048,
        num_workers=2,
        max_test_samples=1048575,
    )
else:
    cross_metrics = None
    if KDD_DATA_PATH is None:
        print('Skipping — KDD dataset not found.')
    else:
        print('Skipping — CICIDS model not trained yet (run cells 4-5 first).')

GPU 0: Tesla T4 (14.6 GB)
GPU 1: Tesla T4 (14.6 GB)
Model trained on 78 features (input_dim=78)
Using threshold: 0.7090
Loading target dataset: /kaggle/input/datasets/towhidultonmoy/cicids2018/Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
Target dataset: 1048575 samples, 78 features
Attack ratio: 0.5739 (601803 attacks / 1048575 total)

FEATURE MAPPING SUMMARY
Total training features : 78
Mapped features         : 77
  Exact matches         : 27
  Manual alias matches  : 50
  Normalized matches    : 0
Missing / median-filled : 1
Saved mapping report -> /kaggle/working/cross_dataset_kdd/resolved_feature_mapping.csv

Aligned raw target matrix shape: (1048575, 78)


/tmp/ipykernel_55/1085860467.py:50: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)



CROSS-DATASET TEST RESULTS
  Train data features: 78 (Destination Port, Flow Duration, Total Fwd Packets, Total Backward Packets, Total Length of Fwd Packets, Total Length of Bwd Packets, Fwd Packet Length Max, Fwd Packet Length Min, Fwd Packet Length Mean, Fwd Packet Length Std...)
  Test data features:  78 (Dst Port, Protocol, Flow Duration, Tot Fwd Pkts, Tot Bwd Pkts, TotLen Fwd Pkts, TotLen Bwd Pkts, Fwd Pkt Len Max, Fwd Pkt Len Min, Fwd Pkt Len Mean...)
  Mapped features:     77 / 78
  Exact matches:       27
  Alias matches:       50
  Normalized matches:  0
  Missing / filled:    1
  Loss:      1.7594
  Threshold: 0.7090
  ROC-AUC:   0.2511
  PR-AUC:    0.5241
  F1-Score:  0.3764
  Precision: 0.9890
  Recall:    0.2325
  Accuracy:  0.5580
Saved results -> /kaggle/working/cross_dataset_kdd/cross_dataset_results.csv


In [12]:
import os
import glob
import pandas as pd

def evaluate_all_csvs_in_directory(
    checkpoint_path: str,
    data_dir: str,
    output_dir: str,
    batch_size: int = 2048,
    num_workers: int = 2,
    max_test_samples: int = 0,
):
    """
    Run cross_dataset_evaluate on every CSV file in data_dir.
    Each CSV gets its own output subfolder so result files do not overwrite.
    """
    os.makedirs(output_dir, exist_ok=True)

    csv_files = sorted(glob.glob(os.path.join(data_dir, "*.csv")))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in: {data_dir}")

    print(f"Found {len(csv_files)} CSV files:")
    for f in csv_files:
        print(" -", os.path.basename(f))

    all_results = []

    for i, csv_path in enumerate(csv_files, 1):
        file_name = os.path.splitext(os.path.basename(csv_path))[0]
        file_output_dir = os.path.join(output_dir, file_name)
        os.makedirs(file_output_dir, exist_ok=True)

        print("\n" + "=" * 100)
        print(f"[{i}/{len(csv_files)}] Evaluating: {os.path.basename(csv_path)}")
        print("=" * 100)

        try:
            metrics = cross_dataset_evaluate(
                checkpoint_path=checkpoint_path,
                test_data_path=csv_path,
                output_dir=file_output_dir,
                batch_size=batch_size,
                num_workers=num_workers,
                max_test_samples=max_test_samples,
            )

            all_results.append({
                "file_name": file_name,
                "file_path": csv_path,
                **metrics,
            })

        except Exception as e:
            print(f"Failed on {csv_path}: {e}")
            all_results.append({
                "file_name": file_name,
                "file_path": csv_path,
                "error": str(e),
            })

    summary_df = pd.DataFrame(all_results)
    summary_path = os.path.join(output_dir, "all_csv_cross_dataset_summary.csv")
    summary_df.to_csv(summary_path, index=False)

    print("\nSaved combined summary ->", summary_path)
    return summary_df

In [13]:
cnn_path = '/kaggle/input/models/samaraweeramethun/trained-model-2017/pytorch/default/1/cnn_transformer_ids.pth'

CICIDS2018_DIR = '/kaggle/input/datasets/towhidultonmoy/cicids2018'

if cnn_path is not None:
    cross_out = os.path.join(cnn_cfg.output_dir, 'cross_dataset_cicids2018_results')
    os.makedirs(cross_out, exist_ok=True)

    cross_summary_df = evaluate_all_csvs_in_directory(
        checkpoint_path=cnn_path,
        data_dir=CICIDS2018_DIR,
        output_dir=cross_out,
        batch_size=2048,
        num_workers=2,
        max_test_samples=1048575,   # use 0 if you want all rows
    )

    display(cross_summary_df)

else:
    cross_summary_df = None
    print('Skipping — model checkpoint not found.')

Found 10 CSV files:
 - Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
 - Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
 - Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
 - Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
 - Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv
 - Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv
 - Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv
 - Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv
 - Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv
 - Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv

[1/10] Evaluating: Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
GPU 0: Tesla T4 (14.6 GB)
GPU 1: Tesla T4 (14.6 GB)
Failed on /kaggle/input/datasets/towhidultonmoy/cicids2018/Friday-02-03-2018_TrafficForML_CICFlowMeter.csv: [Errno 2] No such file or directory: '/kaggle/input/models/samaraweeramethun/trained-model-2017/pytorch/default/1/cnn_transformer_ids.pth'

[2/10] Evaluating: Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
GPU 0: Tesla T4 (1

,file_name,file_path,error
0,Friday-02-03-2018_TrafficForML_CICFlowMeter,/kaggle/input/datasets/towhidultonmoy/cicids20...,[Errno 2] No such file or directory: '/kaggle/...
1,Friday-16-02-2018_TrafficForML_CICFlowMeter,/kaggle/input/datasets/towhidultonmoy/cicids20...,[Errno 2] No such file or directory: '/kaggle/...
2,Friday-23-02-2018_TrafficForML_CICFlowMeter,/kaggle/input/datasets/towhidultonmoy/cicids20...,[Errno 2] No such file or directory: '/kaggle/...
3,Thuesday-20-02-2018_TrafficForML_CICFlowMeter,/kaggle/input/datasets/towhidultonmoy/cicids20...,[Errno 2] No such file or directory: '/kaggle/...
4,Thursday-01-03-2018_TrafficForML_CICFlowMeter,/kaggle/input/datasets/towhidultonmoy/cicids20...,[Errno 2] No such file or directory: '/kaggle/...
5,Thursday-15-02-2018_TrafficForML_CICFlowMeter,/kaggle/input/datasets/towhidultonmoy/cicids20...,[Errno 2] No such file or directory: '/kaggle/...
6,Thursday-22-02-2018_TrafficForML_CICFlowMeter,/kaggle/input/datasets/towhidultonmoy/cicids20...,[Errno 2] No such file or directory: '/kaggle/...
7,Wednesday-14-02-2018_TrafficForML_CICFlowMeter,/kaggle/input/datasets/towhidultonmoy/cicids20...,[Errno 2] No such file or directory: '/kaggle/...
8,Wednesday-21-02-2018_TrafficForML_CICFlowMeter,/kaggle/input/datasets/towhidultonmoy/cicids20...,[Errno 2] No such file or directory: '/kaggle/...
9,Wednesday-28-02-2018_TrafficForML_CICFlowMeter,/kaggle/input/datasets/towhidultonmoy/cicids20...,[Errno 2] No such file or directory: '/kaggle/...
